In [1]:
import sqlite3
import pandas as pd
import sqlalchemy as sqla
import os

print("SQLite version:", sqlite3.sqlite_version)
print("SQLAlchemy version:", sqla.__version__)

SQLite version: 3.45.1
SQLAlchemy version: 2.1.7


In [2]:
if os.path.exists("mydata.sqlite"):
    os.remove("mydata.sqlite")

con = sqlite3.connect("mydata.sqlite")

print("Database created successfully!")

Database created successfully!


In [3]:
query = """
CREATE TABLE test (
    city VARCHAR(20),
    state VARCHAR(20),
    rainfall REAL,
    rainy_days INTEGER
);
"""

con.execute(query)
con.commit()

print("Table created successfully!")

Table created successfully!


In [4]:
data = [
    ("Atlanta", "Georgia", 1.25, 6),
    ("Tallahassee", "Florida", 2.6, 3),
    ("Sacramento", "California", 1.7, 5)
]

stmt = "INSERT INTO test VALUES (?, ?, ?, ?)"

con.executemany(stmt, data)
con.commit()

print("3 rows inserted!")

3 rows inserted!


In [5]:
cursor = con.execute("SELECT * FROM test")

rows = cursor.fetchall()

print(rows)

[('Atlanta', 'Georgia', 1.25, 6), ('Tallahassee', 'Florida', 2.6, 3), ('Sacramento', 'California', 1.7, 5)]


In [6]:
col_names = [x[0] for x in cursor.description]

print(col_names)

['city', 'state', 'rainfall', 'rainy_days']


In [7]:
df_manual = pd.DataFrame(rows, columns=col_names)

print(df_manual)

          city       state  rainfall  rainy_days
0      Atlanta     Georgia      1.25           6
1  Tallahassee     Florida      2.60           3
2   Sacramento  California      1.70           5


In [8]:
db = sqla.create_engine("sqlite:///mydata.sqlite")

print("Database connected successfully!")

Database connected successfully!


In [9]:
df = pd.read_sql("SELECT * FROM test", db)

print(df)

          city       state  rainfall  rainy_days
0      Atlanta     Georgia      1.25           6
1  Tallahassee     Florida      2.60           3
2   Sacramento  California      1.70           5


In [10]:
print(df.dtypes)

city           object
state          object
rainfall      float64
rainy_days      int64
dtype: object


In [11]:
data3 = [
    ("Mumbai", "Maharashtra", 9.8, 20),
    ("Chennai", "Tamil Nadu", 4.2, 12),
    ("Jaipur", "Rajasthan", 1.1, 4),
    ("Kolkata", "West Bengal", 6.5, 18),
    ("Orlando", "Florida", 3.1, 9)
]

con.executemany(
    "INSERT INTO test VALUES (?, ?, ?, ?)",
    data3
)

con.commit()

print("Additional rows inserted!")

Additional rows inserted!


In [12]:
df = pd.read_sql("SELECT * FROM test", db)

print(df)

          city        state  rainfall  rainy_days
0      Atlanta      Georgia      1.25           6
1  Tallahassee      Florida      2.60           3
2   Sacramento   California      1.70           5
3       Mumbai  Maharashtra      9.80          20
4      Chennai   Tamil Nadu      4.20          12
5       Jaipur    Rajasthan      1.10           4
6      Kolkata  West Bengal      6.50          18
7      Orlando      Florida      3.10           9


In [13]:
con.execute(
    "INSERT INTO test VALUES (?, ?, ?, ?)",
    ("Orlando", "Florida", 3.1, 9)
)

con.commit()

print("Single row inserted!")

Single row inserted!


In [14]:
df = pd.read_sql(
    "SELECT city, rainfall FROM test",
    db
)

print(df)

          city  rainfall
0      Atlanta      1.25
1  Tallahassee      2.60
2   Sacramento      1.70
3       Mumbai      9.80
4      Chennai      4.20
5       Jaipur      1.10
6      Kolkata      6.50
7      Orlando      3.10
8      Orlando      3.10


In [15]:
df = pd.read_sql(
    "SELECT * FROM test WHERE rainfall > 3",
    db
)

print(df)

      city        state  rainfall  rainy_days
0   Mumbai  Maharashtra       9.8          20
1  Chennai   Tamil Nadu       4.2          12
2  Kolkata  West Bengal       6.5          18
3  Orlando      Florida       3.1           9
4  Orlando      Florida       3.1           9


In [16]:
df = pd.read_sql(
    "SELECT city, rainfall FROM test ORDER BY rainfall DESC",
    db
)

print(df)

          city  rainfall
0       Mumbai      9.80
1      Kolkata      6.50
2      Chennai      4.20
3      Orlando      3.10
4      Orlando      3.10
5  Tallahassee      2.60
6   Sacramento      1.70
7      Atlanta      1.25
8       Jaipur      1.10


In [17]:
df = pd.read_sql("""
SELECT
    state,
    AVG(rainfall) AS avg_rainfall,
    COUNT(*) AS num_cities
FROM test
GROUP BY state
ORDER BY avg_rainfall DESC
""", db)

print(df)

         state  avg_rainfall  num_cities
0  Maharashtra      9.800000           1
1  West Bengal      6.500000           1
2   Tamil Nadu      4.200000           1
3      Florida      2.933333           3
4   California      1.700000           1
5      Georgia      1.250000           1
6    Rajasthan      1.100000           1


In [19]:
students = pd.DataFrame({
    "roll_no": [101, 102, 103, 104],
    "name": ["Aman", "Bhavna", "Chirag", "Divya"],
    "marks": [78.5, 85.0, 62.0, 91.5]
})

print(students)

   roll_no    name  marks
0      101    Aman   78.5
1      102  Bhavna   85.0
2      103  Chirag   62.0
3      104   Divya   91.5


In [20]:
students.to_sql(
    "students",
    db,
    if_exists="replace",
    index=False
)

print("Students table saved successfully!")

Students table saved successfully!


In [21]:
df = pd.read_sql(
    "SELECT * FROM students",
    db
)

print(df)

   roll_no    name  marks
0      101    Aman   78.5
1      102  Bhavna   85.0
2      103  Chirag   62.0
3      104   Divya   91.5


In [22]:
df = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'",
    db
)

print(df)

       name
0      test
1  students


In [23]:
db.dispose()
con.close()

print("Database connection closed!")

Database connection closed!


In [24]:
if os.path.exists("mydata.sqlite"):
    os.remove("mydata.sqlite")

print("Database deleted successfully!")

Database deleted successfully!
